<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Llama 3.1 70B와 Ollama를 사용하여 선호도 데이터셋 생성하기

- 선호도 미세조정(preference finetuning)은 지시 미세조정된 LLM을 인간의 선호도와 정렬하는 과정입니다
- LLM의 선호도 미세조정을 위한 데이터셋을 생성하는 방법은 여러 가지가 있습니다:
  1. 지시 미세조정된 LLM을 사용하여 여러 응답을 생성하고, 인간이 선호도 및/또는 주어진 선호도 기준에 따라 순위를 매기도록 합니다
  2. 지시 미세조정된 LLM을 사용하여 여러 응답을 생성하고, LLM이 주어진 선호도 기준에 따라 순위를 매기도록 합니다
  3. LLM을 사용하여 특정 선호도 기준에 따라 선호되는 응답과 선호되지 않는 응답을 생성합니다
- 이 노트북에서는 접근법 3을 고려합니다
- 이 노트북은 ollama를 통해 700억 파라미터의 Llama 3.1-Instruct 모델을 사용하여 지시 데이터셋에 대한 선호도 라벨을 생성합니다
- 지시 데이터셋의 예상 형식은 다음과 같습니다:


### 입력

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

출력 데이터셋은 다음과 같이 보이며, 더 정중한 응답이 선호(`'chosen'`)되고, 더 무례한 응답이 비선호(`'rejected'`)됩니다:

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### 출력




- 이 코드는 GPU가 필요하지 않으며 충분한 RAM이 있는 노트북에서 실행됩니다

In [1]:
from importlib.metadata import version

pkgs = ["tqdm",    # 진행률 표시줄
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## Ollama 설치 및 Llama 3.1 다운로드

- Ollama는 LLM을 효율적으로 실행하는 애플리케이션입니다
- 이는 효율성을 극대화하기 위해 순수 C/C++로 LLM을 구현한 [llama.cpp](https://github.com/ggerganov/llama.cpp)를 감싸는 래퍼입니다
- 이것은 텍스트 생성(추론)을 위한 LLM 사용 도구이며, LLM 학습이나 미세조정을 위한 것이 아님에 주목하세요
- 아래 코드를 실행하기 전에 [https://ollama.com](https://ollama.com)을 방문하여 지침을 따라 ollama를 설치하세요 (예: "Download" 버튼을 클릭하고 운영 체제용 ollama 애플리케이션을 다운로드)

- macOS 및 Windows 사용자의 경우 다운로드한 ollama 애플리케이션을 클릭하세요. 명령줄 사용을 설치하라는 메시지가 표시되면 "yes"라고 답하세요
- Linux 사용자는 ollama 웹사이트에 제공된 설치 명령어를 사용할 수 있습니다

- 일반적으로 명령줄에서 ollama를 사용하기 전에 ollama 애플리케이션을 시작하거나 별도의 터미널에서 `ollama serve`를 실행해야 합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- ollama 애플리케이션이나 `ollama serve`가 실행 중인 상태에서 다른 터미널의 명령줄에서 다음 명령을 실행하여 700억 파라미터의 Llama 3.1 모델을 시도해보세요

```bash
# 70B 모델
ollama run llama3.1:70b
```


출력은 다음과 같습니다:

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- `llama3.1:70b`는 지시 미세조정된 700억 파라미터 Llama 3.1 모델을 가리킵니다

- 또는 `llama3.1:70b`를 `llama3.1`로 바꾸어 더 작고 리소스 효율적인 80억 파라미터 Llama 3.1 모델을 사용할 수도 있습니다

- 다운로드가 완료된 후 모델과 채팅할 수 있는 명령줄 프롬프트가 나타납니다

- "What do llamas eat?"와 같은 프롬프트를 시도해보세요. 다음과 비슷한 출력을 반환해야 합니다:

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- `/bye` 입력을 사용하여 이 세션을 종료할 수 있습니다

## Ollama의 REST API 사용

- 이제 모델과 상호작용하는 대안적인 방법은 다음 함수를 통해 Python의 REST API를 사용하는 것입니다
- 이 노트북의 다음 셀들을 실행하기 전에 위에서 설명한 대로 ollama가 여전히 실행 중인지 확인하세요:
  - 터미널에서 `ollama serve`
  - ollama 애플리케이션
- 다음으로 모델을 쿼리하기 위해 다음 코드 셀을 실행하세요

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [2]:
import urllib.request
import json


def query_model(prompt, model="llama3.1:70b", url="http://localhost:11434/api/chat"):
    # 데이터 페이로드를 딕셔너리로 생성
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
        }
    }

    # 딕셔너리를 JSON 형식의 문자열로 변환하고 바이트로 인코딩
    payload = json.dumps(data).encode("utf-8")

    # 요청 객체를 생성하고 메서드를 POST로 설정하며 필요한 헤더 추가
    request = urllib.request.Request(url, data=payload, method="POST")
    request.add_header("Content-Type", "application/json")

    # 요청을 보내고 응답을 캡처
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # 응답을 읽고 디코딩
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet consists of:

1. **Grasses**: Various types of grasses, including timothy grass, orchard grass, and brome grass.
2. **Hay**: High-quality hay, such as alfalfa or clover hay, is a staple in a llama's diet.
3. **Leaves**: Leaves from trees and shrubs, like willow, cottonwood, and mesquite, are also eaten.
4. **Fruits and vegetables**: Llamas enjoy fruits like apples, carrots, and sweet potatoes, as well as leafy greens like kale and spinach.
5. **Grains**: In moderation, llamas can eat grains like oats, barley, and corn.

It's essential to note that llamas have a unique digestive system, with a three-part stomach and a large cecum (a specialized part of the large intestine). This allows them to break down and extract nutrients from plant material more efficiently than many other animals.

A typical llama diet might consist of:

* 1-2% of their body weight in hay per day
* 0.5-1% of their body w

## JSON 항목 로드

- 이제 데이터 생성 부분으로 넘어가겠습니다
- 여기서 실습 예제를 위해 7장에서 모델을 지시 미세조정할 때 원래 사용했던 `instruction-data.json` 파일을 사용합니다:

In [3]:
from pathlib import Path

json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 1100


- 이 파일의 구조는 다음과 같습니다. 여기서 `'input'`과 `'instruction'`을 기반으로 지시 미세조정을 통해 모델이 생성하도록 학습한 테스트 데이터셋의 주어진 응답(`'output'`)이 있습니다

In [4]:
json_data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

- 아래는 지시와 입력을 형식화하는 작은 유틸리티 함수입니다:

In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 이제 ollama API를 시도해서 모델 선호도 튜닝을 위한 `'chosen'`과 `'rejected'` 응답을 생성해보겠습니다
- 여기서는 설명 목적으로 더 정중하거나 덜 정중한 답변을 만듭니다


In [6]:
import random


for entry in json_data[:5]:
    
    politeness = random.choice(["polite", "impolite"])    
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print(f"\n{politeness} response:")
    print(">>", query_model(prompt))    


Dataset response:
>> The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".

impolite response:
>> The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".

Dataset response:
>> He goes to the park every day.

polite response:
>> He goes to the park daily, if I'm not mistaken.

Dataset response:
>> 45 kilometers is 45000 meters.

polite response:
>> 45 kilometers is equivalent to 45000 meters.

Dataset response:
>> Although it was raining, they went for a walk.

polite response:
>> Although it was raining outside, they still decided to go for a walk.

Dataset response:
>> 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.

impolite response:
>> Here are your precious square numbers: 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.


- 위에서 생성된 응답이 합리적으로 보인다면 다음 단계로 넘어가서 전체 데이터셋에 프롬프트를 적용할 수 있습니다
- 여기서는 선호되는 응답을 위한 `'chosen'` 키와 선호되지 않는 응답을 위한 `'rejected'` 응답을 추가합니다

In [7]:
import random
from tqdm import tqdm

def generate_model_responses(json_data):

    for i, entry in enumerate(tqdm(json_data, desc="Writing entries")):
        politeness = random.choice(["polite", "impolite"])    
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        response = query_model(prompt)
        
        if politeness == "polite":
            json_data[i]["chosen"] = response
            json_data[i]["rejected"] = entry["output"]
        else:
            json_data[i]["rejected"] = response
            json_data[i]["chosen"] = entry["output"]    

- 이제 이 평가를 전체 데이터셋에 적용하고 각 모델의 평균 점수를 계산해보겠습니다 (M3 MacBook Air 노트북에서 모델당 약 1분 소요)
- ollama는 (이 글을 쓰는 시점에서) 운영 체제 간에 완전히 결정론적이지 않으므로 아래 표시된 것과 약간 다른 숫자가 나올 수 있습니다

In [8]:
generate_model_responses(json_data)

Writing entries: 100%|██████████| 1100/1100 [17:20<00:00,  1.06it/s]


In [9]:
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)